# Stage 1 — eval results vs. the papers

Interactive companion to `scripts/03_eval.sh`. Loads what that script (and
`load_hf_adapter.py`, if some adapters only live on the HF Hub) produced under
`runs/<run name>/`, plots it, and compares it against real numbers pulled from both papers'
PDFs (see the **Paper reference values** cell below for exactly where each number comes from —
arXiv:2603.01204 Tables 1/4/5, arXiv:2606.00995 Figures 2/3/7c).

Run this with the **project-root** `uv` env (`uv run jupyter lab` from the repo root), not
either vendored repo's venv — this notebook only needs torch/transformers/peft/pandas/
matplotlib/ipywidgets/huggingface_hub for reading already-computed results and (optionally)
downloading adapters; it doesn't run vLLM generation itself.

**Pick a run and traits below, then re-run the "render" cell (or just change the widgets — it
re-renders automatically on any change, no manual re-run needed).** Defaults to the paper's own
traits (`cat`/`lion`/`panda`) since that's the direct-comparison subset — see
`TRAIT_SUBSET=cat,lion,panda ./run_all.sh` in the README for running just those three.

In [ ]:
import json
import sys
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent  # rlhf/stage1_subliminal_traits
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

ALL_TRAITS = ["cat", "lion", "panda", "dog", "octopus", "oak", "willow", "birch"]
PAPER_TRAITS = ["cat", "lion", "panda"]  # arXiv:2603.01204's own targets — direct comparison
EAS_TRAITS_AVAILABLE = ["cat", "panda"]  # only these ever get step-checkpoint EAS data

## Pick a run and traits

In [ ]:
run_name_box = widgets.Text(value="deepjudge_s1", description="RUN_NAME:", style={"description_width": "initial"})
traits_select = widgets.SelectMultiple(
    options=ALL_TRAITS, value=tuple(PAPER_TRAITS), description="traits:", style={"description_width": "initial"},
    layout=widgets.Layout(width="220px", height="160px"),
)
download_button = widgets.Button(description="Download missing adapters from HF Hub", layout=widgets.Layout(width="280px"))
download_output = widgets.Output()


def _on_download_click(_):
    from load_hf_adapter import load_one

    download_output.clear_output()
    with download_output:
        run_dir = PROJECT_ROOT / "runs" / run_name_box.value
        for trait in ["neutral", *traits_select.value]:
            load_one(run_dir, trait)


download_button.on_click(_on_download_click)

display(widgets.HBox([run_name_box, traits_select]))
display(download_button, download_output)

## Results

Everything below re-renders automatically whenever `RUN_NAME` or the trait selection above
changes. Prefers each run's `eval/summary.csv` (written by `summarize_eval.py` at the end of
`03_eval.sh`); falls back to reading the per-trait JSON/`.pt` files directly if that hasn't
been generated yet, so this works even mid-run.

In [ ]:
def _read_json(path: Path):
    return json.loads(path.read_text()) if path.exists() else None


def _cos_at_extract_layer(path: Path):
    if not path.exists():
        return None
    meta = torch.load(path, map_location="cpu", weights_only=False)["meta"]
    if meta["extract_layer"] is None:
        return None
    return meta["cos_v_teacher_per_layer"][meta["extract_layer"] + 1]


def load_results(run_dir: Path, traits: list[str]) -> pd.DataFrame:
    summary_csv = run_dir / "eval" / "summary.csv"
    if summary_csv.exists():
        full = pd.read_csv(summary_csv)
        return full[full["trait"].isin(traits)].reset_index(drop=True)

    rows = []
    for trait in traits:
        trait_dir = run_dir / "eval" / trait
        own = _read_json(trait_dir / "own" / f"{trait}_own_eval" / "eval_results.json")
        neutral = _read_json(trait_dir / "neutral_control" / f"{trait}_neutral_control_eval" / "eval_results.json")
        cos = _cos_at_extract_layer(run_dir / "vectors" / f"v_student_{trait}.pt")
        eas = _read_json(trait_dir / "eas.json")
        rows.append(
            {
                "trait": trait,
                "own_target_rate": own["cat_rate"] if own else None,
                "neutral_control_target_rate": neutral["cat_rate"] if neutral else None,
                "lift_over_neutral": (own["cat_rate"] - neutral["cat_rate"]) if own and neutral else None,
                "cos_v_student_v_teacher_at_extract_layer": cos,
                "eas_at_layer_final": eas["main_curve"][-1]["eas_at_layer"] if eas and eas.get("main_curve") else None,
            }
        )
    return pd.DataFrame(rows)

In [ ]:
def render(run_name: str, traits: tuple[str, ...]):
    if not traits:
        print("select at least one trait above")
        return

    run_dir = PROJECT_ROOT / "runs" / run_name
    traits = list(traits)
    print(f"RUN_DIR = {run_dir}  (exists: {run_dir.exists()})")

    df = load_results(run_dir, traits)
    display(df)

    # Plot 1 — target rate: own adapter vs. neutral control.
    fig, ax = plt.subplots(figsize=(max(6, 1.1 * len(traits)), 5))
    x = range(len(df))
    width = 0.35
    own = df["own_target_rate"].fillna(0)
    neutral = df["neutral_control_target_rate"].fillna(0)
    if df.empty or df["own_target_rate"].isna().all():
        print("no target-rate results yet (own_target_rate is all-NaN) — run 03_eval.sh first; plotting zeros as a placeholder")
    ax.bar([i - width / 2 for i in x], own, width, label="own DPO adapter", color="#3b6ba5")
    ax.bar([i + width / 2 for i in x], neutral, width, label="neutral-DPO control", color="#bbbbbb")
    ax.set_xticks(list(x))
    ax.set_xticklabels(df["trait"] if not df.empty else [])
    ax.set_ylabel("target-word rate (SVD sl-eval, freeform, 50 prompts)")
    ax.set_title(f"Target-rate per trait — own adapter vs. neutral control ({run_name})")
    ax.legend()
    ax.set_ylim(0, 1)
    fig.tight_layout()
    plt.show()

    # Plot 2 — activation-diff cosine alignment.
    fig, ax = plt.subplots(figsize=(max(6, 1.1 * len(traits)), 5))
    cos_vals = df["cos_v_student_v_teacher_at_extract_layer"].fillna(0) if not df.empty else []
    if df.empty or df["cos_v_student_v_teacher_at_extract_layer"].isna().all():
        print("no activation-diff results yet — run 03_eval.sh first; plotting zeros as a placeholder")
    colors = ["#c96a3a" if t in PAPER_TRAITS else "#5a9e6f" for t in df["trait"]] if not df.empty else []
    ax.bar(df["trait"] if not df.empty else [], cos_vals, color=colors)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("cos(v_student, v_teacher) at extract layer")
    ax.set_title("Activation-diff alignment per trait (orange = paper's own targets)")
    fig.tight_layout()
    plt.show()

    # Plot 3 — EAS_n emergence (only for whichever of cat/panda were selected).
    eas_traits = [t for t in traits if t in EAS_TRAITS_AVAILABLE]
    if eas_traits:
        fig, axes = plt.subplots(1, len(eas_traits), figsize=(6 * len(eas_traits), 5), sharey=True)
        if len(eas_traits) == 1:
            axes = [axes]
        for ax, trait in zip(axes, eas_traits):
            eas = _read_json(run_dir / "eval" / trait / "eas.json")
            if eas is None:
                ax.set_title(f"{trait}: no eas.json yet")
                continue
            main = eas["main_curve"]
            control = eas.get("control_curve") or []
            ax.plot([r["step"] for r in main], [r["eas_at_layer"] for r in main], label=f"{trait} (biased judge)", color="#3b6ba5")
            if control:
                ax.plot([r["step"] for r in control], [r["eas_at_layer"] for r in control], label="neutral control", color="#bbbbbb")
            ax.axhline(0.1, color="gray", linestyle=":", linewidth=0.8, label="paper's control plateau (~0.1)")
            ax.set_xscale("log")
            ax.set_xlabel("training step")
            ax.set_title(f"EAS_n — {trait}")
            ax.legend()
        axes[0].set_ylabel("cos(h, v_teacher)")
        fig.tight_layout()
        plt.show()

    # Side-by-side vs. paper win-rates, for whichever of cat/lion/panda were selected.
    common = [t for t in traits if t in PAPER_TRAITS]
    if common:
        ours = df.set_index("trait").reindex(common)
        theirs = paper1_dpo.set_index("trait").reindex(common)
        lift = ours["lift_over_neutral"].fillna(0)
        if ours["lift_over_neutral"].isna().all():
            print(f"no results yet for {common} — run 03_eval.sh first; plotting zeros as a placeholder")

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        ax1.bar(common, lift, color="#3b6ba5")
        ax1.set_title("Ours: target-rate lift over neutral control\n(SVD sl-eval, freeform)")
        ax1.set_ylabel("own_target_rate - neutral_control_target_rate")
        ax1.axhline(0, color="black", linewidth=0.8)

        ax2.bar(common, theirs["paper_win_rate_normal_vs_control_pct"], color="#c96a3a")
        ax2.set_title("Paper: DPO win-rate, normal vs. control\n(arXiv:2603.01204 Table 5, MC logprob eval)")
        ax2.set_ylabel("win rate (%)")
        ax2.set_ylim(0, 100)

        fig.suptitle("Same qualitative claim (DPO transmits the trait), different metrics — not directly comparable magnitudes")
        fig.tight_layout()
        plt.show()

## Paper reference values

Extracted directly from the two PDFs (`arXiv:2603.01204` and `arXiv:2606.00995`) via `pypdf`,
not from memory — see the exact table/figure cited on each line. **Two important caveats before
comparing these to the plots above:**

1. arXiv:2603.01204's own headline numbers use a **multiple-choice logprob eval** ("pick your
   favorite animal from A-E") and **win-rate / log-prob-difference** metrics — not the same
   metric as SVD's `sl-eval` **freeform target-word rate** our `own_target_rate` column reports.
   Both should move in the same direction if the effect replicated, but the magnitudes aren't
   directly on the same scale.
2. arXiv:2606.00995's own numbers (EAS_n, v_student sufficiency/necessity, Fig 7c hit-rate) come
   from **LoRA SFT distillation on teacher-generated data**, not DPO on judge preference labels
   — a different training pipeline than this repo's stage 1. Treat these as an order-of-magnitude
   / qualitative-pattern reference ("does EAS rise the same way, does activation-diff show the
   same sign"), not a number our DPO run should be expected to match exactly.

In [ ]:
# arXiv:2603.01204 (ETH-DISCO), DPO / deep judge, Table 1 & Table 4 & Table 5.
paper1_dpo = pd.DataFrame(
    [
        # trait, win_rate_normal_vs_swapped_%, win_rate_normal_vs_control_%, effect_size_logprob_units
        ("cat", 82.000, 80.000, 13.34),
        ("lion", 96.000, 96.000, 13.24),
        ("panda", 52.000, 44.000, 1.36),
    ],
    columns=["trait", "paper_win_rate_normal_vs_swapped_pct", "paper_win_rate_normal_vs_control_pct", "paper_effect_size"],
)
print("arXiv:2603.01204 Table 1 / Table 4 / Table 5 -- DPO, deep judge, Qwen2.5-7B-Instruct")
paper1_dpo

In [ ]:
# arXiv:2606.00995 (SVD), qualitative reference points (SFT-distillation pipeline, Qwen2.5-7B-Instruct
# unless noted). No single scalar for these -- recorded as the ranges/statements the paper reports.
paper2_reference = {
    "EAS_n control plateau (Fig 2)": "~0.1 cos(h, v_teacher) for a clean/neutral-data-trained student, across traits",
    "EAS_n biased trend (Fig 2, cat)": "rises from ~0 toward ~0.2-0.3 over the first ~1000 training steps",
    "v_student sufficiency (Fig 3a, Qwen)": "steering the reference model with v_student raises trait hit-rate toward the LoRA student's own rate",
    "v_student necessity (Fig 3b, Qwen)": "replacing the student's v_student component with the reference model's removes over 50% of trait-aligned behavior",
    "Optimizer ablation (Fig 7c, cat, Qwen LoRA)": "AdamW reaches the highest cat hit-rate among optimizers tested; plain SGD fails to induce it",
}
for k, v in paper2_reference.items():
    print(f"- {k}: {v}")

## Render\n\nClick **Update plots** after changing the run name or trait selection above -- each click appends a fresh set of plots below (not wrapped in an `Output()` widget on purpose: that combination hangs under headless execution, e.g. `jupyter nbconvert --execute`; a plain cell output keeps this notebook runnable both live and headless). Renders once with the defaults as soon as this cell runs, then again on every click.

In [ ]:
update_button = widgets.Button(description="Update plots", button_style="primary")


def _update(_=None):
    render(run_name_box.value, traits_select.value)


update_button.on_click(_update)
display(update_button)
_update()